# T2.4 – DBRepo View Definitions

Creates five named views in the group database via the DBRepo REST API.
The database and all tables must already exist (created in T2.1).

**Requires**: database owner or `write_all` access for view creation.
If the database is not accessible the notebook prints the reason and stops.

| View | Filter | Purpose |
|------|--------|---------|
| `ml_accident_features` | — | Full de-normalised feature table (all accidents) |
| `ml_fatal_accidents` | severity_id = 1 | Fatal accidents — minority-class subset |
| `ml_serious_accidents` | severity_id = 2 | Serious accidents — minority-class subset |
| `ml_rural_accidents` | rural_urban = 'Rural' | Rural-area accidents |
| `ml_high_speed_accidents` | speed_limit_mph ≥ 60 | High-speed road accidents |

In [1]:
# %pip install dbrepo==1.13.5 requests

In [2]:
import json
import sys
import time
import requests
from pathlib import Path
from requests.auth import HTTPBasicAuth

In [3]:
ENDPOINT     = "https://test.dbrepo.tuwien.ac.at"
USERNAME     = ""
PASSWORD     = ""
CONTAINER_ID = "6cfb3b8e-1792-4e46-871a-f3d103527203"

IDS_FILE = Path("dbrepo_ids.json")
ids  = json.loads(IDS_FILE.read_text())
DB_ID   = ids["database_id"]
DB_URL  = f"{ENDPOINT}/api/v1/database/{DB_ID}"
AUTH    = HTTPBasicAuth(USERNAME, PASSWORD)
HEADERS = {"Content-Type": "application/json"}

print("Database ID:", DB_ID)

Database ID: f36ef3e2-1aee-4526-b3ea-82f661a9261a


## 1 · Check database access

In [4]:
r = requests.get(DB_URL, auth=AUTH)
if r.status_code != 200:
    print(f"[ERROR] Cannot access database {DB_ID}")
    print(f"  HTTP {r.status_code}: {r.json().get('message', r.text)}")
    print("\nView creation requires the database owner to grant access.")
    print("Ask the database owner to run:")
    print(f"  POST {ENDPOINT}/api/v1/database/{DB_ID}/access/Chrisvenator")
    print("  Body: {\"type\": \"write_all\"}")
    sys.exit(0)

db = r.json()
print(f"Connected: {db['name']}")
print(f"  owner:     {db.get('contact', {}).get('username', '?')}")
print(f"  is_public: {db['is_public']}  is_schema_public: {db['is_schema_public']}")
print(f"  tables:    {len(db.get('tables', []))}")

Connected: rta_stats19_north_yorkshire
  owner:     icedex
  is_public: True  is_schema_public: True
  tables:    19


## 2 · Resolve column and operator UUIDs

In [5]:
col_map, table_map = {}, {}
for tbl in db.get("tables", []):
    iname = tbl["internal_name"]
    table_map[iname] = tbl["id"]
    for c in tbl.get("columns", []):
        col_map[f"{iname}.{c['internal_name']}"] = c["id"]

container = requests.get(f"{ENDPOINT}/api/v1/container/{CONTAINER_ID}", auth=AUTH).json()
OP = {o["value"]: o["id"] for o in container["image"]["operators"]}

print(f"Tables indexed:   {len(table_map)}")
print(f"Columns indexed:  {len(col_map)}")
print(f"Operators loaded: {list(OP.keys())}")

Tables indexed:   19
Columns indexed:  69
Operators loaded: ['=', '<=>', '<', '<=', '>', '>=', '!=', 'LIKE', 'NOT LIKE', 'IN', 'NOT IN', 'IS NOT NULL', 'IS NULL', 'REGEXP', 'NOT REGEXP']


## 3 · Shared column list and join chain

All five views use the same 25-column projection and the same three-level
geographic join chain:
```
accident → output_area → lower_super_output_area → local_authority_district
```
The `oa11_code` join key is excluded from the SELECT to avoid duplicate column
names. Column UUIDs are resolved from the live schema, so the payload is always
current. Operator UUIDs come from the MariaDB image definition.

In [6]:
BASE_COLS_NAMES = [
    "accident.police_ref",       "accident.accident_date",       "accident.accident_time",
    "accident.day_of_week",      "accident.easting",             "accident.northing",
    "accident.longitude",        "accident.latitude",            "accident.severity_id",
    "accident.road_cond_id",     "accident.light_condition_id",  "accident.weather_condition_id",
    "accident.special_condition_id", "accident.carriageway_hazard_id", "accident.road_type_id",
    "accident.speed_limit_mph",  "accident.junction_detail_id",  "accident.junction_control_id",
    "accident.crossing_control_id", "accident.crossing_facility_id",
    "accident.casualties",       "accident.vehicles",
    "output_area.rural_urban",   "output_area.area_hectares",
    "local_authority_district.lad_name",
]
SELECT_COLS = [{"id": col_map[c]} for c in BASE_COLS_NAMES]

BASE_JOINS = [
    {"type": "inner", "datasource_id": table_map["output_area"],
     "conditionals": [{"column_id":         col_map["accident.oa11_code"],
                        "foreign_column_id": col_map["output_area.oa11_code"]}]},
    {"type": "inner", "datasource_id": table_map["lower_super_output_area"],
     "conditionals": [{"column_id":         col_map["output_area.lsoa_id"],
                        "foreign_column_id": col_map["lower_super_output_area.lsoa_id"]}]},
    {"type": "inner", "datasource_id": table_map["local_authority_district"],
     "conditionals": [{"column_id":         col_map["lower_super_output_area.lad12_code"],
                        "foreign_column_id": col_map["local_authority_district.lad12_code"]}]},
]

print(f"SELECT columns: {len(SELECT_COLS)}")
print(f"JOIN steps:     {len(BASE_JOINS)}")

SELECT columns: 25
JOIN steps:     3


In [7]:
def upsert_view(name, filters=None):
    """Delete-then-create so the notebook is idempotent."""
    for v in requests.get(f"{DB_URL}/view", auth=AUTH).json():
        if v["name"] == name:
            requests.delete(f"{DB_URL}/view/{v['id']}", auth=AUTH)
            print(f"  deleted existing '{name}'")
            break
    payload = {
        "name": name,
        "query": {
            "columns":        SELECT_COLS,
            "datasource_ids": [table_map["accident"]],
            "joins":          BASE_JOINS,
            "filters":        filters or [],
        },
        "is_public": True, "is_schema_public": True,
    }
    r = requests.post(f"{DB_URL}/view", auth=AUTH, json=payload, headers=HEADERS)
    if r.status_code == 201:
        v = r.json()
        print(f"  created '{v['name']}'  id={v['id']}")
        return v["id"]
    print(f"  [ERROR] {name}: HTTP {r.status_code}: {r.json().get('message', r.text[:200])}")
    return None


def flt(col_name, operator, value):
    return {"type": "where", "column_id": col_map[col_name],
            "operator_id": OP[operator], "value": value}

## 4 · Create the five views

In [8]:
print("View 1: ml_accident_features")
v1_id = upsert_view("ml_accident_features")

print("View 2: ml_fatal_accidents")
v2_id = upsert_view("ml_fatal_accidents",     [flt("accident.severity_id",    "=",  "1")])

print("View 3: ml_serious_accidents")
v3_id = upsert_view("ml_serious_accidents",   [flt("accident.severity_id",    "=",  "2")])

print("View 4: ml_rural_accidents")
v4_id = upsert_view("ml_rural_accidents",     [flt("output_area.rural_urban", "=",  "Rural")])

print("View 5: ml_high_speed_accidents")
v5_id = upsert_view("ml_high_speed_accidents",[flt("accident.speed_limit_mph",">=", "60")])

View 1: ml_accident_features


  [ERROR] ml_accident_features: HTTP 403: Failed to create view: not the database owner
View 2: ml_fatal_accidents


  [ERROR] ml_fatal_accidents: HTTP 403: Failed to create view: not the database owner
View 3: ml_serious_accidents


  [ERROR] ml_serious_accidents: HTTP 403: Failed to create view: not the database owner
View 4: ml_rural_accidents


  [ERROR] ml_rural_accidents: HTTP 403: Failed to create view: not the database owner
View 5: ml_high_speed_accidents


  [ERROR] ml_high_speed_accidents: HTTP 403: Failed to create view: not the database owner


## 5 · Save IDs and verify

In [9]:
created = {k: v for k, v in {
    "ml_accident_features":    v1_id,
    "ml_fatal_accidents":      v2_id,
    "ml_serious_accidents":    v3_id,
    "ml_rural_accidents":      v4_id,
    "ml_high_speed_accidents": v5_id,
}.items() if v is not None}

if created:
    ids["views"] = {**ids.get("views", {}), **created}
    IDS_FILE.write_text(json.dumps(ids, indent=2))
    print(f"Saved {len(created)} view ID(s) to {IDS_FILE}")

time.sleep(4)
print("\nViews in DBRepo:")
for v in requests.get(f"{DB_URL}/view", auth=AUTH).json():
    print(f"  {v['name']}  id={v['id']}")


Views in DBRepo:
